In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..') if '__file__' in dir() else os.path.abspath('..'))

import torch
from rl.curricula import Curriculum
from rl.policy import Policy
from rl.ppo import train_epoch
from tqdm import tqdm

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

policy = Policy(window_radius=7).to(device)
optimizer = torch.optim.Adam(policy.parameters(), lr=2.5e-4, eps=1e-5)

In [3]:
stages = [
    Curriculum(num_envs=1024, num_agents=1, terr_delta_coef=0.5,  death_penalty=-10.0, env_size=15, leave_terr_bonus=1),
    Curriculum(num_envs=256, num_agents=2, env_size=30, death_penalty=-10.0),
    Curriculum(num_envs=64, death_penalty=-5.0),
]

stage = 0
curriculum = stages[stage]

best_reward = -float('inf')
stall_count = 0
patience = 20

os.makedirs("checkpoints", exist_ok=True)

pbar = tqdm(range(5000))
for iteration in pbar:
    metrics = train_epoch(curriculum, policy, optimizer, device=device)
    reward = metrics['mean_reward']

    if reward > best_reward:
        best_reward = reward
        stall_count = 0
        torch.save(policy.state_dict(), f"checkpoints/stage_{stage + 1}_best.pt")
    else:
        stall_count += 1

    stage_str = f"stg {stage + 1}/{len(stages)}"
    pbar.set_postfix_str(
        f"loss={metrics['policy_loss']:.3f} "
        f"v_loss={metrics['value_loss']:.3f} "
        f"ent={metrics['entropy']:.3f} "
        f"rew={reward:.3f} "
        f"ev={metrics['explained_variance']:.2f} "
        f"stall={stall_count}/{patience} {stage_str}"
    )

    if stall_count >= patience:
        stage += 1
        if stage >= len(stages):
            print("All stages complete.")
            break
        print(f"\n>>> Advancing to stage {stage + 1}")
        curriculum = stages[stage]
        best_reward = -float('inf')
        stall_count = 0

  1%|          | 57/5000 [07:51<11:20:51,  8.26s/it, loss=-0.002 v_loss=55.759 ent=0.548 rew=128.816 ev=0.86 stall=4/20 stg 1/3] 


KeyboardInterrupt: 